# 🎯 Module 08: RNN — Recurrent Neural Networks

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Recurrent Neural Networks (RNNs)  
> **Framework:** PyTorch  
> **Date:** 08 June 2026  
> **Previous:** 07 — CNNs  
> **Next:** 09 — LSTM & GRU

---

Up to now, our networks have treated each input **independently**. But much of the world is *sequential*:
text, speech, stock prices, sensor readings, weather data, logs, user activity.

**RNNs** introduce **memory** — a hidden state that carries information forward through time.

In this notebook we'll build RNNs from the math up to a working classifier and time-series regressor.

We will cover:

- Why sequence data needs a new architecture
- The hidden state and the core RNN equation
- Parameter sharing across time
- RNN tensor shapes in PyTorch
- `nn.RNN` — outputs vs hidden state
- Many-to-one, one-to-many, many-to-many
- Backpropagation Through Time (BPTT)
- Vanishing & exploding gradients + gradient clipping
- Bidirectional & stacked RNNs
- Building a full RNN classifier and regressor

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain **why** RNNs exist and when to use them
- Describe the role of the **hidden state**
- Write the core RNN equation from memory
- Compute RNN parameter counts by hand
- Understand PyTorch `output` vs `hidden`
- Distinguish **many-to-one**, **one-to-many**, **many-to-many** setups
- Explain BPTT, vanishing and exploding gradients
- Apply **gradient clipping**
- Use `batch_first=True` correctly
- Build, train and evaluate an RNN classifier and regressor
- Explain why **LSTM / GRU** were invented

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Is an RNN?

An **RNN (Recurrent Neural Network)** is a neural network designed for **sequential data**.

Unlike a feed-forward network, an RNN keeps a **hidden state** that carries information from previous time steps.

### Sequential data examples

- Text and sentences
- Speech
- Time series / stock prices
- Sensor readings
- Weather measurements
- User activity / logs

### The core idea

```text
Previous information
        ↓
Current input → RNN → Current output
        ↑
     hidden state
```

At every time step, the network receives:

```text
current input + previous hidden state
```

and produces:

```text
new hidden state + output
```

---
## ❓ 2. Why Do We Need RNNs?

Consider:

```text
I went to the bank to deposit money.
```

The meaning of `bank` depends on surrounding words — a dense network sees the whole sentence as a flat feature vector and has no notion of *order* or *context*.

An RNN processes the sequence:

```text
I → went → to → the → bank → to → deposit → money
```

and carries context forward in its **hidden state** — a small *memory*.

### Feed-forward vs RNN

| Feature | Feed-forward | RNN |
| --- | --- | --- |
| Sequential input | Not natural | ✅ Yes |
| Memory | ❌ None | ✅ Hidden state |
| Order matters | Usually not | ✅ Yes |
| Text / time series | Limited | Designed for it |
| Variable sequence length | Less natural | ✅ Natural |

---
## 🧮 3. The Core RNN Equation

At time `t`:

| Symbol | Meaning |
| --- | --- |
| $x_t$ | current input |
| $h_{t-1}$ | previous hidden state |
| $h_t$ | new hidden state |
| $y_t$ | output |

### Hidden state update

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

### Output

$$y_t = W_{hy} h_t + b_y$$

The **most important** equation:

$$\boxed{h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)}$$

### Unrolled through time

```text
x₁       x₂       x₃       x₄
↓        ↓        ↓        ↓
┌───┐   ┌───┐   ┌───┐   ┌───┐
│RNN│ → │RNN│ → │RNN│ → │RNN│
└───┘   └───┘   └───┘   └───┘
 ↓        ↓        ↓        ↓
h₁       h₂       h₃       h₄
```

The **same parameters** are reused at every time step — this is **parameter sharing**.

---
## 🔢 4. A Manual RNN Step (By Hand)

Suppose:

$$x_t = 2,\quad h_{t-1} = 0.5,\quad W_{xh} = 0.4,\quad W_{hh} = 0.6,\quad b = 0.1$$

Then:

$$z_t = (0.4)(2) + (0.6)(0.5) + 0.1 = 0.8 + 0.3 + 0.1 = 1.2$$

$$h_t = \tanh(1.2) \approx 0.834$$

The new hidden state is approximately **0.834**.

In [ ]:
# Verify the manual step
x_t   = 2.0
h_prev = 0.5
W_xh = 0.4
W_hh = 0.6
b     = 0.1

z = W_xh * x_t + W_hh * h_prev + b
h_t = np.tanh(z)
print(f'z_t = {z:.4f}')
print(f'h_t = tanh(z_t) = {h_t:.4f}')

---
## 🎢 5. Why `tanh`?

`tanh` maps any real number into the range $[-1, 1]$:

```text
large positive → close to +1
0              → 0
large negative → close to -1
```

This gives the hidden state a **bounded representation**.

⚠️ However, `tanh` **saturates** — its derivative becomes tiny for large magnitude inputs.
This contributes to the **vanishing gradient problem** (Section 15).

In [ ]:
# Visualize tanh and its saturation
z = np.linspace(-6, 6, 400)
plt.figure(figsize=(7, 3.5))
plt.plot(z, np.tanh(z), label='tanh(z)')
plt.plot(z, 1 - np.tanh(z)**2, label="tanh'(z) = 1 - tanh²(z)", linestyle='--')
plt.axhline(0, color='k', linewidth=0.5)
plt.title('tanh and its derivative (saturates for |z| large)')
plt.xlabel('z'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🧮 6. RNN Dimensions & Parameter Count

For `input_size = 3` and `hidden_size = 4`:

$$W_{xh} \in \mathbb{R}^{4 \times 3}, \quad
  W_{hh} \in \mathbb{R}^{4 \times 4}, \quad
  b_h    \in \mathbb{R}^{4}$$

### Parameter count

$$\text{Params} = D_h D_x + D_h D_h + D_h$$

Example (D_x=3, D_h=4):

$$4 \cdot 3 + 4 \cdot 4 + 4 = 12 + 16 + 4 = 32$$

That's tiny — RNNs use **massive parameter sharing** across time.

In [ ]:
def rnn_params(input_size, hidden_size, bias=True):
    p = hidden_size * input_size + hidden_size * hidden_size
    if bias: p += hidden_size
    return p

print('input_size=3,  hidden_size=4 :', rnn_params(3, 4),  'params')
print('input_size=10, hidden_size=20:', rnn_params(10, 20),'params')

# Verify with PyTorch
rnn = nn.RNN(input_size=3, hidden_size=4, batch_first=True)
actual = sum(p.numel() for p in rnn.parameters())
print(f'\nPyTorch actual for (3 → 4): {actual} params')

---
## 📐 7. PyTorch Tensor Shapes — `batch_first`

With `batch_first=True` (recommended):

$$[\text{batch}, \text{sequence}, \text{features}]$$

Example: `[32, 10, 50]` means

```text
32  = batch size
10  = sequence length (time steps)
50  = features per time step
```

⚠️ **Without** `batch_first=True`, PyTorch expects:

$$[\text{sequence}, \text{batch}, \text{features}]$$

This is one of the **most common RNN mistakes**.

In [ ]:
# A complete forward pass through nn.RNN
batch_size   = 4
seq_len      = 6
input_size   = 10
hidden_size  = 16

x = torch.randn(batch_size, seq_len, input_size)   # [B, T, F]
rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)

output, hidden = rnn(x)
print('Input  shape:', x.shape)
print('Output shape:', output.shape, '  → [B, T, H]')
print('Hidden shape:', hidden.shape, '  → [num_layers, B, H]')

---
## 🔍 8. `output` vs `hidden` — What's the Difference?

### `output`
Hidden representation for **every time step**:

$$[\text{batch}, \text{sequence}, \text{hidden}]$$

### `hidden`
**Final** hidden state (for each layer & direction):

$$[\text{num\_layers} \times \text{num\_directions},\ \text{batch},\ \text{hidden}]$$

For a **many-to-one** task (e.g. sentiment classification), we usually use the **last** hidden state:

```python
final_hidden = hidden[-1]   # [B, H]
```

In [ ]:
# Extracting the final hidden state
final_hidden = hidden[-1]
print('Final hidden shape:', final_hidden.shape)  # [4, 16]

# Feed it through a linear classifier
classifier = nn.Linear(hidden_size, 2)   # 2 classes
logits = classifier(final_hidden)
print('Logits shape:      ', logits.shape)   # [4, 2]

# Sanity: output[:, -1, :] should equal final_hidden
print('\noutput[:, -1] vs hidden[-1] equal?',
      torch.allclose(output[:, -1, :], final_hidden, atol=1e-6))

---
## 🔀 9. Sequence-to-Sequence Configurations

### 1. One-to-One
```text
input → output          e.g. image → class (not really RNN)
```

### 2. Many-to-One
```text
x₁ x₂ x₃ x₄ → output    e.g. sentence → sentiment
```

### 3. One-to-Many
```text
input → x₁ x₂ x₃ x₄     e.g. image → caption
```

### 4. Many-to-Many
```text
x₁ → y₁
x₂ → y₂
x₃ → y₃                 e.g. word → POS tag
x₄ → y₄
```

We'll implement **many-to-one** (classification) and **many-to-one regression** in this notebook.

---
## 🏛️ 10. Building an RNN Classifier

Architecture for a many-to-one task:

```text
[B, T, F]
    ↓
   RNN
    ↓
[B, T, H]  and  hidden [1, B, H]
    ↓
final hidden = hidden[-1]  → [B, H]
    ↓
Linear(H → num_classes)
    ↓
[B, num_classes]
```

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        output, hidden = self.rnn(x)
        final_hidden = hidden[-1]          # last layer's hidden state
        return self.fc(final_hidden)


# Sanity check shapes
model = RNNClassifier(input_size=50, hidden_size=64, num_classes=2)
x = torch.randn(32, 20, 50)                 # [B, T, F]
y = model(x)
print('Input :', x.shape)
print('Output:', y.shape)                    # [32, 2]
print('Params:', sum(p.numel() for p in model.parameters()), 'total')

---
## 🏃 11. Training the RNN Classifier

We'll build a **synthetic** sequence classification task:

- Each sequence has length `T`
- Positive class = sequences whose average signal is > 0
- Negative class = the rest

The RNN must learn to **aggregate information across time**.

In [ ]:
# Generate synthetic sequence classification data
torch.manual_seed(42)

N_TRAIN, N_VAL = 2000, 500
T, F = 20, 10   # sequence length, features

def make_seq_data(n):
    X = torch.randn(n, T, F)
    # Signal: mean over time of feature 0 (plus a bit of feature 1)
    signal = X[:, :, 0].mean(dim=1) + 0.5 * X[:, :, 1].mean(dim=1)
    y = (signal > 0).long()
    return X, y

X_train, y_train = make_seq_data(N_TRAIN)
X_val,   y_val   = make_seq_data(N_VAL)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=128, shuffle=False)

print('Train:', X_train.shape, 'Val:', X_val.shape)
print('Positive fraction (train):', y_train.float().mean().item())
print('Positive fraction (val)  :', y_val.float().mean().item())

In [ ]:
# Training + eval helpers
def train_one_epoch(model, loader, optimizer, criterion, clip=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        if clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

print('Helpers ready.')

In [ ]:
# Train the classifier
torch.manual_seed(42)
model = RNNClassifier(input_size=F, hidden_size=64, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, clip=1.0)
    va_loss, va_acc = evaluate(model, val_loader, criterion)

    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | '
          f'val_loss={va_loss:.4f}   val_acc={va_acc:.4f}')

In [ ]:
# Visualize learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 🔁 12. Backpropagation Through Time (BPTT)

RNN parameters are **reused across time**, so gradients must flow backward through the whole sequence:

```text
Forward:   x₁ → x₂ → x₃ → x₄ →  h₁ → h₂ → h₃ → h₄
Backward:  x₁ ← x₂ ← x₃ ← x₄ ←  h₁ ← h₂ ← h₃ ← h₄
```

This is essentially **backprop applied to the unrolled recurrent network**.

### Why BPTT is difficult

With a long sequence (say 100 steps), gradients pass through many multiplications:

- Repeatedly small factors (0.5 × 0.5 × 0.5 × …) → **vanishing gradients**
- Repeatedly large factors (2 × 2 × 2 × …) → **exploding gradients**

---
## 💥 13. Vanishing & Exploding Gradients

### Vanishing gradient
If the gradient is `0.5` and it passes through 10 recurrent steps:

$$0.5^{10} \approx 0.00098$$

The signal becomes **negligible** — the model can't learn long-range dependencies.

### Exploding gradient
If the gradient is `2` across 10 steps:

$$2^{10} = 1024$$

Large gradients cause:

- unstable training
- huge parameter updates
- NaN values
- training failure

In [ ]:
# Visualize the explosion / vanishing of repeated factors
steps = np.arange(1, 21)
plt.figure(figsize=(8, 4))
plt.plot(steps, 0.5 ** steps, 'o-', label='0.5^n  (vanishing)')
plt.plot(steps, 2.0 ** steps, 's-', label='2.0^n  (exploding)')
plt.yscale('log')
plt.xlabel('time steps n'); plt.ylabel('magnitude (log scale)')
plt.title('Repeated multiplication of gradients')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## ✂️ 14. Gradient Clipping

A simple, effective fix for **exploding** gradients:

```python
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

If the total gradient norm exceeds `max_norm`, it's rescaled — capping the update size.

**Rule of thumb:** clip in *every* RNN training loop.

In [ ]:
# Quick demo: gradient clipping in action
tiny = nn.Sequential(nn.Linear(4, 4))
out = tiny(torch.randn(2, 4)).sum()
out.backward()
print('Before clip, grad norm:', sum(p.grad.norm().item() for p in tiny.parameters()))

# Scale grads artificially to huge values, then clip
for p in tiny.parameters():
    p.grad *= 1000

torch.nn.utils.clip_grad_norm_(tiny.parameters(), max_norm=1.0)
print('After clip,  grad norm:', sum(p.grad.norm().item() for p in tiny.parameters()))

---
## ⚠️ 15. Why Vanilla RNNs Struggle with Long-Term Dependencies

Consider:

```text
The students who studied very hard for the difficult exam
because they wanted to pass it eventually ______ .
```

The information needed is **many steps back**. Vanilla RNNs lose this context because:

1. The same hidden state is overwritten at every step
2. Gradients vanish over long sequences

This motivated **LSTM** and **GRU**, which add **gating** mechanisms to control what gets remembered, forgotten, or exposed.

---
## ↔️ 16. Bidirectional RNNs

A **bidirectional RNN** runs two RNNs in opposite directions and concatenates their hidden states:

```text
Forward:  x₁ → x₂ → x₃ → x₄
Backward: x₄ → x₃ → x₂ → x₁
```

```python
nn.RNN(input_size=50, hidden_size=64, batch_first=True, bidirectional=True)
```

Hidden size becomes `64 × 2 = 128`.

### When to use
✅ Full sequence available at prediction time — text classification, NER, POS tagging.  
❌ **Real-time forecasting** — never look into the future (data leakage).

---
## 🧱 17. Stacked (Multi-Layer) RNNs

```python
nn.RNN(input_size=50, hidden_size=64, num_layers=2, dropout=0.2, batch_first=True)
```

More layers → more capacity, but also more complexity and training difficulty.  
`dropout` only applies **between** layers (requires `num_layers > 1`).

---
## 📊 18. RNN vs CNN vs LSTM vs GRU

### RNN vs CNN

| Feature | CNN | RNN |
| --- | --- | --- |
| Main purpose | Spatial patterns | Sequential patterns |
| Images | ✅ Excellent | ❌ Not natural |
| Text / time series | Possible | ✅ Natural |
| Memory | ❌ No | ✅ Hidden state |
| Main dimension | H × W | Time |

### RNN vs LSTM

| Feature | Vanilla RNN | LSTM |
| --- | --- | --- |
| Hidden state | ✅ | ✅ |
| Cell state | ❌ | ✅ |
| Gates | ❌ | ✅ (forget / input / output) |
| Long dependencies | ❌ Difficult | ✅ Better |
| Parameters | Fewer | More |

### RNN vs GRU
GRU = simpler LSTM with **update** and **reset** gates. Fewer parameters, often similar performance.

---
## 📈 19. RNN for Time-Series Regression

Task: predict the **next** value of `sin(t)` from the previous 5 values.

```text
[sin₁, sin₂, sin₃, sin₄, sin₅] → sin₆
[sin₂, sin₃, sin₄, sin₅, sin₆] → sin₇
...
```

Architecture:

```text
5 time steps
     ↓
    RNN
     ↓
final hidden
     ↓
Linear → 1 prediction
```

In [ ]:
# Generate sin(t) windows
t = np.linspace(0, 100, 2000)
series = np.sin(t).astype(np.float32)

WINDOW = 5   # sequence length
X_list, y_list = [], []
for i in range(len(series) - WINDOW):
    X_list.append(series[i:i + WINDOW])
    y_list.append(series[i + WINDOW])

X_all = torch.tensor(np.array(X_list)).unsqueeze(-1)  # [N, T, 1]
y_all = torch.tensor(np.array(y_list)).unsqueeze(-1)  # [N, 1]

n_train = int(0.8 * len(X_all))
X_train_r, y_train_r = X_all[:n_train], y_all[:n_train]
X_val_r,   y_val_r   = X_all[n_train:], y_all[n_train:]

train_loader_r = DataLoader(TensorDataset(X_train_r, y_train_r), batch_size=64, shuffle=True)
val_loader_r   = DataLoader(TensorDataset(X_val_r,   y_val_r),   batch_size=128, shuffle=False)

print('X_train:', X_train_r.shape, ' y_train:', y_train_r.shape)
print('X_val:  ', X_val_r.shape,   ' y_val:  ', y_val_r.shape)

In [ ]:
# Define the RNN regressor
class RNNRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, hidden = self.rnn(x)
        return self.fc(hidden[-1])

print('RNNRegressor ready.')

In [ ]:
# Train the RNN regressor
torch.manual_seed(42)
reg = RNNRegressor(input_size=1, hidden_size=32).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(reg.parameters(), lr=1e-3)

EPOCHS = 20
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    reg.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader_r:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = reg(xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reg.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    train_losses.append(total_loss / n)

    reg.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader_r:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(reg(xb), yb)
            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)
    val_losses.append(total_loss / n)

    print(f'Epoch {epoch:2d}/{EPOCHS} | train_mse={train_losses[-1]:.5f} | val_mse={val_losses[-1]:.5f}')

In [ ]:
# Plot training curves and predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='train')
axes[0].plot(val_losses,   label='val')
axes[0].set_title('MSE loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

reg.eval()
with torch.no_grad():
    preds = reg(X_val_r.to(device)).cpu().squeeze().numpy()
axes[1].plot(y_val_r.squeeze().numpy()[:200], label='truth')
axes[1].plot(preds[:200],                 label='pred', linestyle='--')
axes[1].set_title('Validation predictions vs truth')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 🚨 20. Common RNN Mistakes

### ❌ 1. Wrong tensor shape
With `batch_first=True`, expected shape is `[batch, sequence, features]` — **not** `[sequence, batch, features]`.

### ❌ 2. Confusing hidden size with sequence length
`[32, 20, 64]` → 32 = batch, **20 = sequence**, 64 = hidden.

### ❌ 3. Softmax before CrossEntropyLoss
```python
loss = nn.CrossEntropyLoss()(logits, y)        # ✅
loss = nn.CrossEntropyLoss()(F.softmax(logits, 1), y)   # ❌
```

### ❌ 4. Shuffling time steps
RNNs rely on **order**. Never reorder time steps randomly.

### ❌ 5. Using future information
For forecasting `past → future`, never let future values leak into the input. That's **data leakage**.

### ❌ 6. Forgetting `model.eval()`
Dropout and any BatchNorm-style layers behave differently at inference.

---
## 🏋️ 21. Hands-On Exercises

### Exercise 1 — Shapes
Given batch = 32, sequence = 15, input_size = 20, hidden_size = 64, compute:
- input tensor shape
- RNN `output` shape
- `hidden` shape
- final hidden shape after `hidden[-1]`

### Exercise 2 — Parameter Count
Compute the recurrent parameters for `input_size=10, hidden_size=20`:
$$D_hD_x + D_hD_h + D_h$$

### Exercise 3 — From Scratch (NumPy)
Implement `h_t = tanh(W_xh @ x_t + W_hh @ h_prev + b)` in NumPy and process a random sequence.

### Exercise 4 — Hidden-Size Sweep
Train the regressor with `hidden_size ∈ {16, 32, 64}`. Compare validation MSE.

### Exercise 5 — Add Dropout & Layers
Use `num_layers=2, dropout=0.2`. Does it help?

### Exercise 6 — Bidirectional Classifier
Change `nn.RNN(..., bidirectional=True)` and adapt the classifier input size (hidden × 2). Train and compare.

### Exercise 7 — Gradient Clipping Ablation
Train with `max_norm ∈ {None, 0.5, 1.0, 5.0}`. Which is most stable?

---
## 🎤 22. Interview Questions

### Beginner
1. What is an RNN?
2. What is hidden state?
3. Why does an RNN have memory?
4. What activation is traditionally used in vanilla RNNs?
5. What is sequence length?

### Intermediate
6. What is BPTT?
7. Why do RNNs suffer from vanishing gradients?
8. What causes exploding gradients?
9. How can exploding gradients be controlled?
10. Why are RNN parameters shared?

### Advanced
11. Why does a vanilla RNN struggle with long-term dependencies?
12. What's the difference between `output` and `hidden` in PyTorch?
13. Why are LSTMs better suited to long-term dependencies?
14. What is a bidirectional RNN?
15. When should you **avoid** bidirectional models?

---
## ✅ 23. Knowledge Check

Answer these without looking:

1. What problem does an RNN solve?
2. What is hidden state?
3. Why does the previous hidden state matter?
4. Write the RNN hidden-state equation.
5. What does sequence length mean?
6. What is many-to-one?
7. What is many-to-many?
8. What is BPTT?
9. Why do vanishing gradients happen?
10. What is gradient clipping?
11. What does `batch_first=True` mean?
12. What does PyTorch `output` contain?
13. What does `hidden` contain?
14. What is a bidirectional RNN?
15. Why do we need LSTM/GRU?

---
## 🏁 24. Key Takeaways

```text
RNN
│
├── Designed for sequential data
├── Maintains hidden state
├── Processes one time step at a time
├── Shares parameters across time
├── Supports variable sequence lengths
├── Many-to-one, one-to-many, many-to-many
├── Trained using BPTT
├── Can suffer from vanishing gradients
├── Can suffer from exploding gradients
├── Gradient clipping helps exploding gradients
└── Struggles with long-term dependencies
```

### The core mental model

```text
Current input
     +
Previous hidden state
     ↓
    RNN
     ↓
New hidden state
     ↓
Next time step
```

### The major limitation

```text
Vanilla RNN
     ↓
Long-term dependency problem
     ↓
LSTM / GRU
```

### 🧭 Golden Rule

> **Always clip gradients when training RNNs, and never let future information leak into a sequence model that's supposed to predict it.**

---

### 🔗 What's Next?

**Module 09 — LSTM & GRU**

We just proved that a vanilla RNN can't preserve information over long sequences. Next we'll see how **gating** solves that:

```text
RNN → Long-term dependency problem
     ↓
LSTM
 ├── Forget gate
 ├── Input gate
 ├── Output gate
 └── Cell state

GRU
 ├── Update gate
 └── Reset gate
```